# COVID-19 Global Deaths: A First Attempt at Regression

After getting comfortable pulling and cleaning the COVID dataset, I wanted to try something more ambitious: could a plain linear regression predict deaths from case counts? This notebook has two parts. The first fits a simple regression on global data across the whole pandemic timeline. The second narrows in on South Korea over a short window and tries to predict a week ahead.

I am including both parts as they were, warts and all, because the second part taught me more about what *not* to do than the first part taught me about what to do. See the closing notes for what I would fix.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


## Part 1: Global deaths over time

Data comes from Our World in Data's COVID-19 dataset. I filtered to the `World` aggregate row, kept the date, total deaths, and total cases columns, and treated any missing values as zero.


In [ ]:
csv_url = 'https://covid.ourworldindata.org/data/owid-covid-data.csv'

death_df = pd.read_csv(csv_url)

print(death_df[death_df.location == 'World'].tail())


In [ ]:
death_df.date = pd.to_datetime(death_df.date)
death_df = death_df[death_df.location == 'World'][['date', 'total_deaths','total_cases']]
death_df.fillna(0, inplace=True)
print(death_df.date.dtype)
print(death_df.head())


A quick plot of the raw trend before modeling anything.


In [ ]:
import matplotlib.dates as mdates 
import datetime
import matplotlib.ticker as tck

fig, axis = plt.subplots(figsize = (10, 5))
plt.xticks(rotation=50)
axis.xaxis.set_major_locator(mdates.MonthLocator())
axis.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
plt.plot(death_df.date, death_df.total_deaths)
plt.xlim([datetime.date(2020, 1, 1), datetime.date(2021, 11, 23)])
plt.title('Global coronavirus deaths over time')
plt.xlabel('Date')
plt.ylabel('Total deaths')
plt.show()


### Fitting a simple linear regression

The idea: since deaths and cases both climb together, does a single feature (total cases) do a reasonable job predicting total deaths across the full history? This is fit and evaluated on the same data, so it is a check of in-sample fit rather than a genuine predictive test. I am flagging that explicitly rather than presenting it as a held-out evaluation.


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error


In [ ]:
model = LinearRegression()
X = death_df[['total_cases']]
model.fit(X, death_df.total_deaths.values.reshape(-1, 1))
preds = model.predict(X)


In [ ]:
fig, axis = plt.subplots(figsize = (10, 5))
plt.xticks(rotation=50)
axis.xaxis.set_major_locator(mdates.MonthLocator())
axis.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
plt.plot(death_df.date, death_df.total_deaths)
plt.plot(death_df.date, preds)
plt.xlim([datetime.date(2020, 1, 1), datetime.date(2021, 11, 23)])
plt.title('Global coronavirus deaths over time together with a Linear Regression prediction model')
plt.xlabel('Date')
plt.ylabel('Total deaths')
plt.legend(['Original', 'Predicted'])
plt.show()


### In-sample error

Mean squared error on raw death counts (values in the millions) is not, by itself, an interpretable number. I am reporting it here because it was the assignment metric, but treat the value below as a relative signal across the two model attempts in this notebook rather than an absolute measure of quality. RMSE, or MSE normalized against the variance of `total_deaths`, would be a more honest read and is on my list of follow-ups.


In [ ]:
print("Mean Squared Error: ",mean_squared_error(death_df.total_deaths.values.reshape(-1, 1), preds))


## Part 2: A tighter test, South Korea one week ahead

The global fit above is not a real forecasting test since it trains and evaluates on the same stretch of time. For a more honest attempt, I picked South Korea, trained on about twelve weeks of daily data (September 1 to November 22, 2021), and tried to predict the following week (November 23 to November 29).


In [ ]:
csv_url = 'https://covid.ourworldindata.org/data/owid-covid-data.csv'

death_df2 = pd.read_csv(csv_url)

train = death_df2[(death_df2.date >= '2021-09-01') & (death_df2.date <= '2021-11-22') & (death_df2.location == 'South Korea')]
test = death_df2[(death_df2.date > '2021-11-22') & (death_df2.date <= '2021-11-29') & (death_df2.location == 'South Korea')]

train.fillna(0, inplace=True)
test.fillna(0, inplace=True)

print(train.head(), len(train))
print(test, len(test))


In [ ]:
print(train.columns)


In [ ]:
train.info()


I threw nearly every numeric column in the dataset at the model here (about 38 features against roughly 80 training rows). In hindsight this is a textbook overfitting setup: far more features than observations, many of them highly correlated with each other (`total_cases` and `total_cases_per_million`, for example, carry almost the same information). I am keeping the feature list as I originally wrote it rather than editing it retroactively, since the result below is more instructive as a lesson than it would be as a polished outcome.


In [ ]:
X = train[['total_cases', 'new_cases',
       'new_cases_smoothed', 'new_deaths',
       'new_deaths_smoothed', 'total_cases_per_million',
       'new_cases_per_million', 'new_cases_smoothed_per_million',
       'new_deaths_per_million',
       'new_deaths_smoothed_per_million', 'new_tests', 'total_tests',
       'new_tests_per_thousand',
       'new_tests_smoothed', 'new_tests_smoothed_per_thousand',
       'positive_rate', 'tests_per_case',
       'people_vaccinated', 'people_fully_vaccinated', 'total_boosters',
       'new_vaccinations', 'new_vaccinations_smoothed',
       'people_vaccinated_per_hundred',
       'people_fully_vaccinated_per_hundred', 'total_boosters_per_hundred',
       'new_vaccinations_smoothed_per_million',
       'new_people_vaccinated_smoothed',
       'new_people_vaccinated_smoothed_per_hundred','population', 'population_density', 'median_age', 'aged_65_older',
       'aged_70_older', 'gdp_per_capita', 'extreme_poverty',
       'cardiovasc_death_rate', 'diabetes_prevalence','handwashing_facilities', 'hospital_beds_per_thousand',
       'life_expectancy']]
Xtest = test[['total_cases', 'new_cases',
       'new_cases_smoothed', 'new_deaths',
       'new_deaths_smoothed', 'total_cases_per_million',
       'new_cases_per_million', 'new_cases_smoothed_per_million', 
       'new_deaths_per_million',
       'new_deaths_smoothed_per_million', 'new_tests', 'total_tests',
       'new_tests_per_thousand',
       'new_tests_smoothed', 'new_tests_smoothed_per_thousand',
       'positive_rate', 'tests_per_case',
       'people_vaccinated', 'people_fully_vaccinated', 'total_boosters',
       'new_vaccinations', 'new_vaccinations_smoothed',
       'people_vaccinated_per_hundred',
       'people_fully_vaccinated_per_hundred', 'total_boosters_per_hundred',
       'new_vaccinations_smoothed_per_million',
       'new_people_vaccinated_smoothed',
       'new_people_vaccinated_smoothed_per_hundred','population', 'population_density', 'median_age', 'aged_65_older',
       'aged_70_older', 'gdp_per_capita', 'extreme_poverty',
       'cardiovasc_death_rate', 'diabetes_prevalence','handwashing_facilities', 'hospital_beds_per_thousand',
       'life_expectancy']]


model.fit(X, train.total_deaths.values.reshape(-1, 1))
pred = model.predict(Xtest)


### Predicted versus actual, one week ahead


In [ ]:
plt.figure(figsize = (10, 5))
plt.plot(test.date, test.total_deaths)
plt.plot(test.date, pred)
plt.legend(['Original', 'Predicted'])
plt.title('Predicted COVID19 deaths in South Korea for November 23rd to November 29th together with the real values.')
plt.xlabel('Date')
plt.ylabel('Total deaths')
plt.show()


### Test-set error


In [ ]:
print("Mean Squared Error: ",mean_squared_error(test.total_deaths.values.reshape(-1, 1), pred))


## Closing notes

The South Korea test-set MSE is much lower than the global in-sample MSE, but that comparison is not meaningful on its own, the two are on completely different scales of total deaths. What I actually take away from this notebook:

- A single-feature linear regression fit and evaluated on the same data (Part 1) tells you about correlation, not predictive power. I would not present that MSE as a model quality metric without a held-out set.
- The South Korea model (Part 2) used far more features than training rows, several of them redundant. It happened to produce a low test MSE on one seven-day window, which is not strong evidence the approach generalizes. A naive baseline (predicting each day's deaths as the last known value, or a simple moving average) is missing here and would be the right next step before trusting this result.
- Neither part reports RMSE or a normalized error, which would make the numbers easier to interpret at a glance.

Future work: add a naive baseline for comparison, reduce the South Korea feature set to a handful of non-redundant predictors (or use regularization such as Ridge/Lasso given the feature-to-row ratio), and report RMSE alongside MSE.
